In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter

import numpy as np

In [65]:
data_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(32),
        transforms.Normalize((0.5,),(1.0,))             # 간이 정규화 (원랜 이렇게 하면 좋은건 아님)
    ]
)

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=data_transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=data_transform)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Resize(size=32, interpolation=bilinear, max_size=None, antialias=True)
               Normalize(mean=(0.5,), std=(1.0,))
           )

In [66]:
train_data.data.shape

torch.Size([60000, 28, 28])

In [67]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

In [68]:
next(iter(train_loader))[0].shape

torch.Size([128, 1, 32, 32])

In [71]:
class LeNet5(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = F.tanh(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.tanh(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.tanh(self.conv3(x))
        x = x.view(-1,120)              # flatten 형태 잘 맞추기가 가장 중요
        x = F.tanh(self.fc1(x))
        x = F.tanh(self.fc2(x))
        
        return x



model = LeNet5()
model

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)

In [96]:
num_epochs = 10
lr = 1e-3

In [97]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=lr)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)

In [98]:
summary(model, input_size=(1, 64, 64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 60, 60]             156
            Conv2d-2           [-1, 16, 26, 26]           2,416
            Conv2d-3            [-1, 120, 9, 9]          48,120
            Linear-4                   [-1, 84]          10,164
            Linear-5                   [-1, 10]             850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 0.32
Params size (MB): 0.24
Estimated Total Size (MB): 0.57
----------------------------------------------------------------


In [ ]:
writer = SummaryWriter()

count = 0

# checkpoint 불러오기

for ep in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, leave=True)
    for X, y in loop:
        X, y = X.to("cuda"), y.to("cuda")
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        writer.add_scalar("Loss/train", loss, count)
        count += 1
        running_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch [{ep+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train_loader)
    print(f"\nEpoch {ep+1} 평균 Loss: {epoch_loss:.4f}")

  0%|          | 0/469 [00:00<?, ?it/s]

Epoch [1/10]: 100%|██████████| 469/469 [00:33<00:00, 14.09it/s, loss=0.797]



Epoch 1 평균 Loss: 0.8002


Epoch [2/10]: 100%|██████████| 469/469 [00:30<00:00, 15.62it/s, loss=0.797]



Epoch 2 평균 Loss: 0.7999


...


Epoch 7 평균 Loss: 0.7995


Epoch [8/10]: 100%|██████████| 469/469 [00:32<00:00, 14.41it/s, loss=0.797]



Epoch 8 평균 Loss: 0.7991


Epoch [9/10]:  18%|█▊        | 85/469 [00:05<00:25, 15.16it/s, loss=0.797]


KeyboardInterrupt: 

In [ ]:
# due to deprecation of setuptools, you need to downgrade its version <82.0.0 then run, for example:
!uv pip install setuptools==80.10.2
!tensorboard --logdir=runs

Audited 1 package in 8ms


^C


In [100]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for img, y in test_loader:
        img, y = img.to(device), y.to(device)
        
        logits = model(img)
        _, y_pred = torch.max(logits, dim=1)
        
        correct += (y_pred==y).sum().item()
        total += y.size(0)

print(f"test acc: {correct/total}")

test acc: 0.9887


In [101]:
F.softmax(logits)

C:\Users\user\AppData\Local\Temp\ipykernel_14156\383899727.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(logits)


tensor([[0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.4509],
        [0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0

# 학습된 모델 저장방법들

## 방법1 - 권장

In [ ]:
# model 가중치만 저장하는 방법 1 - 가장 권장되고 많이 쓰임.
torch.save(model.state_dict(), "./models/LeNet_CNN_params.pth")

In [107]:
model_new = LeNet5()
model_new.load_state_dict(torch.load("./models/LeNet_CNN_params.pth"))

<All keys matched successfully>

## 방법2 - 보안에 취약함 (deprecated)
* 모델은 저장되는 순간 구조를 알 수 없거나 거의 불가능. 유추는 가능

In [ ]:
# 모델 저장 방법 2
# 모델구조+가중치 통째로 저장하는 방법.
## 바로 불러올 수 없음. 모델 로드 파일에서 설명 cont.
## summary()를 실행한 뒤 저장이 안됨.
torch.save(model_new,"./models/LeNet_CNN_all.pth")

## 방법3 - 가장 권장

In [ ]:
# 모델 저장 방법 3
# checkpoint 활용방법
torch.save({
    "epoch": ep,
    "model_state_dict": model.state_dict(),
    "model_config": {
        "lr": lr,
        "num_epochs": num_epochs,
        "batch_size": 128
    },
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss
}, "./models/checkpoint.pth")

checkpoint = torch.load("./models/checkpoint.pth")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
epoch = checkpoint["epoch"]
loss = checkpoint["loss"]